# T2: DisasTeller on the local model (roadmap T2 steps 1–3)

This notebook runs the whole DisasTeller crew with **no Gemini**: the team's fine-tuned
Qwen2.5-VL-7B runs on this machine with vLLM, and the four agents and three image tools ask it
through `local_model.py`. It then checks the run against the pass rules that the README fixed
before any run ("DisasTeller on the local model").

**Inputs (+ Add Input):**
1. the `offline-rag-kr2` dataset (the guideline search index, as in the day-1 check);
2. the `disasteller-offline` dataset (made by `make_disasteller_upload.py`);
3. the output of `t2_prep_packages.ipynb` (installer files);
4. the output of `t2_prep_model.ipynb` (the model).

**Settings:** Accelerator **GPU T4 x2**.
- **Practice run:** Internet **on**. Everything is still installed and loaded only from the inputs.
- **Proof run:** Internet **off**, then Save Version → **Save & Run All (Commit)**.

**Outputs** (in `t2_run/`, also zipped): the six reports and the labelled map (`run/reports/offline/`),
`run/netlog.json` (every network connection DisasTeller made), `run/run_log.txt`,
`run/llm_calls.jsonl` (every model call: its size in tokens and the model's answer),
`run/t2_run_summary.json` (the pass rules), `vllm_server.log` and `t2_notebook_info.json`.

**5 Nov 2026, after practice run 2:** the model's limit is 32,768 tokens (16,384 in run 2, where one
message grew past it), and DisasTeller now shortens a conversation before it gets too long. The README
section "Practice run 2 and the fixes" explains every change.

In [1]:
import glob, json, os, shutil, socket, subprocess, sys, threading, time, urllib.request, zipfile

SERVED_NAME = "disasterm3-7b"   # the model's name on the server = LOCAL_LLM_MODEL
MAX_MODEL_LEN = 32768           # tokens, question + answer = LOCAL_LLM_CONTEXT (16384 in practice run 2: too small)
PORT = 8000
PACKAGES = ["vllm==0.25.1", "crewai==1.15.2"]   # the same as in t2_prep_packages
RUN_TIMEOUT_HOURS = 4           # stop DisasTeller if it runs longer (for example, a model in a loop)
WORK = "/kaggle/working/t2_run"
os.makedirs(WORK, exist_ok=True)
T0 = time.time()


def minutes():
    return f"{(time.time() - T0) / 60:5.1f} min"

In [2]:
try:
    socket.create_connection(("huggingface.co", 443), timeout=5).close()
    INTERNET = "on"
    print("Internet is ON: this is the practice run. For the proof, turn it off and run again.")
except OSError:
    INTERNET = "off"
    print("Internet is OFF: nothing below can download anything. This is the proof run.")

Internet is ON: this is the practice run. For the proof, turn it off and run again.


Find the four inputs. If Kaggle kept an uploaded zip as a zip, it is unpacked into `/kaggle/working/unzipped/`.

In [3]:
def find_dir(marker, what, zip_name=None):
    hits = sorted(glob.glob(f"/kaggle/input/**/{marker}", recursive=True))
    if not hits and zip_name:
        for z in glob.glob(f"/kaggle/input/**/{zip_name}", recursive=True):
            with zipfile.ZipFile(z) as f:
                f.extractall(f"/kaggle/working/unzipped/{os.path.basename(z)[:-4]}")
        hits = sorted(glob.glob(f"/kaggle/working/unzipped/**/{marker}", recursive=True))
    if not hits:
        raise FileNotFoundError(f"{what} is missing. Add it with + Add Input.")
    return os.path.dirname(hits[0])


RAG_DIR = find_dir("guideline_index.py", "The offline-rag-kr2 dataset", "offline-rag-kr2*.zip")
DT_INPUT = find_dir("run_with_netlog.py", "The disasteller-offline dataset", "disasteller-offline*.zip")
MODEL_DIR = find_dir("t2_model_source.json", "The output of t2_prep_model.ipynb")
WHEELS_DIR = find_dir("t2_wheels_manifest.json", "The output of t2_prep_packages.ipynb")
for name in ("RAG_DIR", "DT_INPUT", "MODEL_DIR", "WHEELS_DIR"):
    print(f"{name:10} {globals()[name]}")

code = json.load(open(f"{DT_INPUT}/disasteller_manifest.json"))
model_source = json.load(open(f"{MODEL_DIR}/t2_model_source.json"))
print("\nDisasTeller code: packed", code["made"], "| git commit", code["git_commit"],
      "| uncommitted changes:", code["uncommitted_changes_in_disasteller_offline"])
print("Model:", model_source["repo"], "| commit", model_source["commit"])

RAG_DIR    /kaggle/input/datasets/riditaafrin/offline-rag-kr2
DT_INPUT   /kaggle/input/datasets/riditaafrin/disasteller-offline1
MODEL_DIR  /kaggle/input/notebooks/riditaafrin/t2-prep-model/model
WHEELS_DIR /kaggle/input/notebooks/riditaafrin/t2-prep-packages/wheels

DisasTeller code: packed 2026-11-08 01:14:20 | git commit 0024ae8af | uncommitted changes: False
Model: AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP | commit def8f6c81889ffb68904afdc90c3a1389697d73b


Put the DisasTeller files where its code expects them, before anything uses the GPU.
Kaggle can change the folder layout of an uploaded dataset: in practice run 1 (3 Nov), the code was
there but `Disaster_image/Local_image/Picture1.png` was not. So each file listed in
`disasteller_manifest.json` is found by its name, checked against its sha256, and copied to
`/kaggle/working/disasteller/` at the path the manifest gives. DisasTeller then runs from that copy.

In [4]:
import hashlib


def sha256(path):
    with open(path, "rb") as f:
        return hashlib.sha256(f.read()).hexdigest()


DT_DIR = "/kaggle/working/disasteller"
shutil.rmtree(DT_DIR, ignore_errors=True)
FOUND_AT, missing = {}, []
for rel, digest in sorted(code["sha256"].items()):
    name = os.path.basename(rel)
    candidates = sorted(glob.glob(f"/kaggle/input/**/{name}", recursive=True)
                        + glob.glob(f"/kaggle/working/unzipped/**/{name}", recursive=True))
    match = next((c for c in candidates if sha256(c) == digest), None)
    if match is None:
        missing.append(f"{rel} (files with this name: {candidates or 'none'})")
        continue
    dest = os.path.join(DT_DIR, rel)
    os.makedirs(os.path.dirname(dest), exist_ok=True)
    shutil.copy2(match, dest)
    FOUND_AT[rel] = match

for rel, path in FOUND_AT.items():
    print(f"{rel:45} <- {path}")
if missing:
    raise FileNotFoundError("Missing or changed in the disasteller-offline dataset: " + "; ".join(missing)
                            + ". Upload disasteller-offline.zip again, as a new version of the dataset.")
shutil.copy2(f"{DT_INPUT}/disasteller_manifest.json", DT_DIR)
print(f"\nAll {len(FOUND_AT)} files are in {DT_DIR}, each with the sha256 in the manifest.")

DisasterManagement_teamwork_simulation.py     <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/DisasterManagement_teamwork_simulation.py
Disaster_image/Global_image/2.jpg             <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Global_image/2.jpg
Disaster_image/Local_image/Picture1.png       <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Picture1.png
Disaster_image/Local_image/Picture2.png       <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Picture2.png
Disaster_image/Local_image/Picture3.png       <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Picture3.png
Disaster_image/Local_image/Picture4.png       <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Picture4.png
Disaster_image/Local_image/Picture5.png       <- /kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Pi

In [5]:
try:
    gpus = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                          capture_output=True, text=True).stdout.strip().splitlines()
except FileNotFoundError:
    gpus = []
print(gpus)
assert len(gpus) >= 2, "Set Accelerator: GPU T4 x2. The 7B is split across two GPUs."

['Tesla T4, 15360 MiB', 'Tesla T4, 15360 MiB']


Install vLLM and CrewAI **from the installer files only** (`--no-index`: pip never looks online).

In [6]:
t = time.time()
r = subprocess.run([sys.executable, "-m", "pip", "install", "--no-index", "--find-links", WHEELS_DIR, *PACKAGES],
                   capture_output=True, text=True)
with open(f"{WORK}/pip_install_log.txt", "w") as f:
    f.write(r.stdout + "\n" + r.stderr)
print(r.stdout[-1500:])
if r.returncode != 0:
    print(r.stderr[-3000:])
    raise RuntimeError("pip install failed: see t2_run/pip_install_log.txt")

VERSIONS = json.loads(subprocess.run([sys.executable, "-c", (
    "import json, importlib.metadata as m\n"
    "v = {}\n"
    "for p in ('vllm', 'crewai', 'torch', 'transformers', 'openai', 'numpy', 'pillow'):\n"
    "    try: v[p] = m.version(p)\n"
    "    except Exception: v[p] = None\n"
    "print(json.dumps(v))")], capture_output=True, text=True).stdout)
print(f"Installed in {(time.time() - t) / 60:.1f} min:", VERSIONS)

.3 lance-namespace-0.13.0 lance-namespace-urllib3-client-0.13.0 lancedb-0.30.0 lark-1.2.2 llguidance-1.7.6 llvmlite-0.47.0 lm-format-enforcer-0.11.3 loguru-0.7.3 mcp-1.26.0 mistral_common-1.13.1 model-hosting-container-standards-0.1.16 msgspec-0.22.0 numba-0.65.0 nvidia-cuda-nvcc-13.2.86 nvidia-cuda-tileiras-13.2.86 nvidia-cudnn-frontend-1.28.0 nvidia-cutlass-dsl-4.5.2 nvidia-cutlass-dsl-libs-base-4.5.2 nvidia-cutlass-dsl-libs-cu13-4.5.2 nvidia-nvvm-13.2.86 nvtx-0.2.15 onnxruntime-1.31.1 openai-harmony-0.0.8 opentelemetry-exporter-otlp-1.42.1 opentelemetry-exporter-otlp-proto-common-1.42.1 opentelemetry-exporter-otlp-proto-grpc-1.42.1 opentelemetry-exporter-otlp-proto-http-1.42.1 opentelemetry-proto-1.42.1 opentelemetry-semantic-conventions-ai-0.5.1 outlines_core-0.2.14 partial-json-parser-0.2.1.1.post7 pdfminer.six-20260107 pdfplumber-0.11.10 pendulum-3.3.0 portalocker-2.7.0 posthog-5.4.0 prometheus-fastapi-instrumentator-8.1.0 protobuf-6.33.6 pybase64-1.5.1 pycountry-26.2.16 pydantic

The model's chat template (how a conversation is laid out for the model). If the model folder has
none, Qwen2.5-VL's standard template is used. It has no section for tools, which is why
DisasTeller uses CrewAI's text format for tool calls (`LOCAL_LLM_NATIVE_TOOLS=0`).

In [7]:
QWEN25_VL_TEMPLATE = "{% set image_count = namespace(value=0) %}{% set video_count = namespace(value=0) %}{% for message in messages %}{% if loop.first and message['role'] != 'system' %}<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n{% endif %}<|im_start|>{{ message['role'] }}\n{% if message['content'] is string %}{{ message['content'] }}<|im_end|>\n{% else %}{% for content in message['content'] %}{% if content['type'] == 'image' or 'image' in content or 'image_url' in content %}{% set image_count.value = image_count.value + 1 %}{% if add_vision_id %}Picture {{ image_count.value }}: {% endif %}<|vision_start|><|image_pad|><|vision_end|>{% elif content['type'] == 'video' or 'video' in content %}{% set video_count.value = video_count.value + 1 %}{% if add_vision_id %}Video {{ video_count.value }}: {% endif %}<|vision_start|><|video_pad|><|vision_end|>{% elif 'text' in content %}{{ content['text'] }}{% endif %}{% endfor %}<|im_end|>\n{% endif %}{% endfor %}{% if add_generation_prompt %}<|im_start|>assistant\n{% endif %}"

config = json.load(open(f"{MODEL_DIR}/config.json"))
print("Architecture:", config.get("architectures"), "| dtype:", config.get("torch_dtype") or config.get("dtype"))
CHAT_TEMPLATE_ARGS = []
if model_source.get("chat_template_in"):
    print("Chat template: the model's own, in", model_source["chat_template_in"])
else:
    path = f"{WORK}/qwen2_5_vl_chat_template.jinja"
    with open(path, "w") as f:
        f.write(QWEN25_VL_TEMPLATE)
    CHAT_TEMPLATE_ARGS = ["--chat-template", path]
    print("Chat template: Qwen2.5-VL's standard one, written to", path)

Architecture: ['Qwen2_5_VLForConditionalGeneration'] | dtype: float16
Chat template: the model's own, in ['chat_template.jinja']


Start the vLLM server on this machine only (`127.0.0.1`), split across the two T4s, in half
precision, with its usage report off. It allows one picture per request, and pictures up to
1344 × 1344 pixels (the map is 1080 × 1522). Loading the 7B takes several minutes.

In [8]:
vllm_cli = shutil.which("vllm")
cmd = ([vllm_cli, "serve", MODEL_DIR] if vllm_cli
       else [sys.executable, "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL_DIR])
cmd += ["--served-model-name", SERVED_NAME, "--host", "127.0.0.1", "--port", str(PORT),
        "--tensor-parallel-size", "2", "--dtype", "half", "--enforce-eager",
        "--max-model-len", str(MAX_MODEL_LEN), "--gpu-memory-utilization", "0.92", "--max-num-seqs", "8",
        "--limit-mm-per-prompt", json.dumps({"image": 1}),
        "--mm-processor-kwargs", json.dumps({"max_pixels": 1344 * 1344}),
        *CHAT_TEMPLATE_ARGS]
server_env = dict(os.environ, HF_HUB_OFFLINE="1", TRANSFORMERS_OFFLINE="1",
                  VLLM_NO_USAGE_STATS="1", DO_NOT_TRACK="1")
print(" ".join(cmd))
server_log = open(f"{WORK}/vllm_server.log", "w")
server = subprocess.Popen(cmd, stdout=server_log, stderr=subprocess.STDOUT, env=server_env)

ready = False
for _ in range(180):  # up to 30 minutes
    if server.poll() is not None:
        break
    try:
        urllib.request.urlopen(f"http://127.0.0.1:{PORT}/health", timeout=5)
        ready = True
        break
    except OSError:
        time.sleep(10)

log_text = open(f"{WORK}/vllm_server.log", errors="replace").read()
if not ready:
    print(log_text[-5000:])
    raise RuntimeError(
        "vLLM did not start: its log is above and in t2_run/vllm_server.log. If it says the KV cache "
        "can't hold max_model_len tokens, set MAX_MODEL_LEN = 16384 in the first cell and run again "
        "(in practice run 2 the KV cache held 198,176 tokens, so 32768 should fit).")
print(f"[{minutes()}] vLLM is ready")
for line in log_text.splitlines():
    if "KV cache" in line or "concurrency" in line:
        print("  ", line[-160:])

/usr/local/bin/vllm serve /kaggle/input/notebooks/riditaafrin/t2-prep-model/model --served-model-name disasterm3-7b --host 127.0.0.1 --port 8000 --tensor-parallel-size 2 --dtype half --enforce-eager --max-model-len 32768 --gpu-memory-utilization 0.92 --max-num-seqs 8 --limit-mm-per-prompt {"image": 1} --mm-processor-kwargs {"max_pixels": 1806336}
[  6.8 min] vLLM is ready
   (Worker_TP0 pid=140) INFO 11-08 03:38:33 [gpu_worker.py:538] Available KV cache memory: 5.29 GiB
   (EngineCore pid=114) INFO 11-08 03:38:33 [kv_cache_utils.py:2146] GPU KV cache size: 198,144 tokens
   (EngineCore pid=114) INFO 11-08 03:38:33 [kv_cache_utils.py:2147] Maximum concurrency for 32,768 tokens per request: 6.05x


Three short tests through `local_model.py`, the same code DisasTeller uses: text, one picture, and a call through CrewAI.

In [9]:
os.environ.update(LOCAL_LLM_PROVIDER="hosted_vllm", LOCAL_LLM_MODEL=SERVED_NAME,
                  LOCAL_LLM_BASE_URL=f"http://127.0.0.1:{PORT}/v1", LOCAL_LLM_CONTEXT=str(MAX_MODEL_LEN),
                  CREWAI_DISABLE_TELEMETRY="true", CREWAI_TRACING_ENABLED="false",
                  CREWAI_DISABLE_VERSION_CHECK="true", LOCAL_LLM_CALL_LOG=f"{WORK}/smoke_llm_calls.jsonl")
sys.path.insert(0, DT_DIR)
import base64, io
from PIL import Image
import local_model

served = json.load(urllib.request.urlopen(f"http://127.0.0.1:{PORT}/v1/models"))
print("Server has:", [m["id"] for m in served["data"]])

buf = io.BytesIO()
Image.open(f"{DT_DIR}/Disaster_image/Local_image/Picture1.png").convert("RGB").resize((720, 720)).save(buf, format="PNG")
photo = base64.b64encode(buf.getvalue()).decode()

SMOKE = {}
for label, ask in [
    ("text", lambda: local_model.ask_with_images("In one sentence: what is a building damage grade?", max_tokens=80)),
    ("picture", lambda: local_model.ask_with_images("Describe the damage in this image in two sentences.", [photo], max_tokens=150)),
    ("crewai", lambda: str(local_model.crew_llm().call([{"role": "user", "content": "Reply with the single word OK."}]))),
]:
    t = time.time()
    SMOKE[label] = answer = ask()
    print(f"{label} ({time.time() - t:.0f} s): {answer[:300]}")
    if answer.startswith("An error occurred"):
        raise RuntimeError(f"The {label} test failed. See t2_run/vllm_server.log.")

Server has: ['disasterm3-7b']
text (6 s): A building damage grade refers to the level of structural damage sustained by a building as a result of an event, such as an earthquake or hurricane.
picture (3 s): The Wajima Drama Memorial Hall has sustained no visible damage as a result of the disaster.
crewai (0 s): OK


**The run.** `run_with_netlog.py` runs DisasTeller and records every network connection it makes.
Settings: offline mode, the Kaggle search backend (`GUIDELINE_BACKEND=numpy`; CrewAI brings chromadb, which
would otherwise pick the Chroma way and try to download its model), and no GPU for DisasTeller itself
(the search model runs on the CPU; the GPUs belong to vLLM). Only key lines are printed here;
everything is in `t2_run/run/run_log.txt`.

In [10]:
RUN_DIR = f"{WORK}/run"
shutil.rmtree(RUN_DIR, ignore_errors=True)
os.makedirs(RUN_DIR)
env = {k: v for k, v in os.environ.items() if k != "DISASTELLER_ONLINE"}
env.update(LOCAL_LLM_PROVIDER="hosted_vllm", LOCAL_LLM_MODEL=SERVED_NAME,
           LOCAL_LLM_BASE_URL=f"http://127.0.0.1:{PORT}/v1", LOCAL_LLM_CONTEXT=str(MAX_MODEL_LEN),
           GUIDELINE_BACKEND="numpy", GUIDELINE_INDEX_DIR=RAG_DIR, HF_HUB_OFFLINE="1",
           TRANSFORMERS_OFFLINE="1", CUDA_VISIBLE_DEVICES="", PYTHONDONTWRITEBYTECODE="1",
           PYTHONIOENCODING="utf-8", PYTHONUNBUFFERED="1",
           LOCAL_LLM_CALL_LOG="llm_calls.jsonl")  # in RUN_DIR (the smoke tests above wrote their own file)
proc = subprocess.Popen([sys.executable, os.path.join(DT_DIR, "run_with_netlog.py")], cwd=RUN_DIR, env=env,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
timer = threading.Timer(RUN_TIMEOUT_HOURS * 3600, proc.kill)
timer.start()
KEY = ("Local model:", "DisasTeller mode", "Task:", "Guideline search:", "Local information (", "Global information:",
       "Labelled map saved", "Could not annotate", "Local model call failed", "Local model gave no answer",
       "Photo tool asked", "Model message shortened", "LLM Call Failed",
       "Maximum iterations reached", "Error parsing LLM output", "An unknown error occurred", "Error details",
       "Error in tool execution", "Context length exceeded",
       "Guideline index loaded", "Guideline search took", "Report finished",
       "Network log", "Outside this computer", "Gemini libraries", "Main script finished", "Traceback")
for line in proc.stdout:
    if any(k in line for k in KEY):
        print(f"[{minutes()}] {line.rstrip()[:220]}")
proc.wait()
timer.cancel()
print(f"[{minutes()}] DisasTeller exit code: {proc.returncode}")

[  7.2 min] Local model: disasterm3-7b at http://127.0.0.1:8000/v1 (hosted_vllm), tool calls: text
[  7.2 min] DisasTeller mode: offline
[  7.2 min] │  Task: Generate descriptions for post-earthquake images using the            │
[  7.2 min] Photo tool asked: describe the photos
[  7.2 min] Local information (Picture1.png): describe the disaster in detail in this image, in about 200 words. If the image has a sticker or sign with a place name, start with that place name. Describe only what this image shows.
[  7.3 min] Local information (Picture2.png): describe the disaster in detail in this image, in about 200 words. If the image has a sticker or sign with a place name, start with that place name. Describe only what this image shows.
[  7.4 min] Local information (Picture3.png): describe the disaster in detail in this image, in about 200 words. If the image has a sticker or sign with a place name, start with that place name. Describe only what this image shows.
[  7.5 min] Local inform

**The pass rules** (fixed in the README before any run), checked by `check_run.py`.

In [11]:
check = subprocess.run([sys.executable, os.path.join(DT_DIR, "check_run.py"), RUN_DIR, RAG_DIR, "--internet", INTERNET],
                       capture_output=True, text=True)
print(check.stdout, check.stderr[-2000:])
for name in ("Expert_team_report.txt", "Reconstruction_plan.txt"):
    path = f"{RUN_DIR}/reports/offline/{name}"
    if os.path.isfile(path):
        print(f"\n===== {name} (first 1500 characters) =====")
        print(open(path, encoding="utf-8", errors="replace").read()[:1500])

PASS  1. The run finished
FAIL  2. All six reports, each starting 'Mode: offline', each 50+ words
PASS  3. Nothing went outside this computer
PASS  4. No Gemini library was loaded
PASS  5. Every model question got an answer

FAILED

report                                             words  cites(real)  'not available offline'
Expert_team_report.txt                                12    0 (  0)    0
Alerts_team_report_for_social_media.txt               82    0 (  0)    0
Emergency_team_report.txt                             17    0 (  0)    0
Emergency_service_human_resource_allocation.txt       14    0 (  0)    0
Public_community_report.txt                           69    0 (  0)    0
Reconstruction_plan.txt                              228    0 (  0)    0

Other signals: {"minutes": 2.6, "network_events": 12, "external": [], "guideline_searches": 3, "photo_questions": 6, "map_questions": 0, "map_label_questions": 1, "labelled_map_saved": true, "map_labels_drawn": [5], "model_messages_s

In [12]:
server.terminate()
try:
    server.wait(timeout=60)
except subprocess.TimeoutExpired:
    server.kill()

info = {
    "internet": INTERNET,
    "minutes": round((time.time() - T0) / 60, 1),
    "served_name": SERVED_NAME,
    "max_model_len": MAX_MODEL_LEN,
    "versions": VERSIONS,
    "model": model_source["repo"],
    "model_commit": model_source["commit"],
    "code_packed": code["made"],
    "code_git_commit": code["git_commit"],
    "code_uncommitted_changes": code["uncommitted_changes_in_disasteller_offline"],
    "code_files_found_at": FOUND_AT,
    "smoke_tests": SMOKE,
    "disasteller_exit_code": proc.returncode,
    "check_run_exit_code": check.returncode,
}
with open(f"{WORK}/t2_notebook_info.json", "w") as f:
    json.dump(info, f, indent=2)
zip_path = shutil.make_archive(f"/kaggle/working/t2_run_internet_{INTERNET}", "zip", WORK)
print(json.dumps({k: v for k, v in info.items() if k != "smoke_tests"}, indent=2))
print("Zip:", zip_path)

{
  "internet": "on",
  "minutes": 9.7,
  "served_name": "disasterm3-7b",
  "max_model_len": 32768,
  "versions": {
    "vllm": "0.25.1",
    "crewai": "1.15.2",
    "torch": "2.11.0+cu130",
    "transformers": "5.20.2",
    "openai": "2.54.0",
    "numpy": "2.1.3",
    "pillow": "13.0.0"
  },
  "model": "AbrarAlam/disasterm3-qwen2.5vl7b-l0s2-mergedFP",
  "model_commit": "def8f6c81889ffb68904afdc90c3a1389697d73b",
  "code_packed": "2026-11-08 01:14:20",
  "code_git_commit": "0024ae8af",
  "code_uncommitted_changes": false,
  "code_files_found_at": {
    "DisasterManagement_teamwork_simulation.py": "/kaggle/input/datasets/riditaafrin/disasteller-offline1/DisasterManagement_teamwork_simulation.py",
    "Disaster_image/Global_image/2.jpg": "/kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Global_image/2.jpg",
    "Disaster_image/Local_image/Picture1.png": "/kaggle/input/datasets/riditaafrin/disasteller-offline1/Disaster_image/Local_image/Picture1.png",
    "Disaster_

**After the run:** download `t2_run_internet_on.zip` or `t2_run_internet_off.zip` from the Output tab into
`Offline_RAG_KR2/results/` (as for the day-1 check), together with the notebook with its outputs.
Never change an output by hand.